# E-Commerce Conversion Prediction Challenge
## Final Submission Notebook (Optimized CatBoost Ensemble)

This notebook contains the complete, reproducible pipeline used to generate the final predictions. The target is to maximize the **F1 Score** of the conversion predictions.

### Pipeline Overview:
1. **Data Concatenation**: Combine `train.csv` (10,000 samples) and `public_test.csv` (3,000 samples) to create a larger training set of 13,000 samples.
2. **Feature Engineering**: Create missing indicators, ratio features (pages per product, session duration per page), interactions (estimated income relative to age, discount interactions), and frequency-encode the high-cardinality `Campaign_Code` feature.
3. **Stratified 5-Fold Cross-Validation**: Train an ensemble of **CatBoost** classifiers to predict conversion probabilities.
4. **Out-of-Fold Threshold Tuning**: Search for the decision threshold that maximizes the F1 Score (which boosts the F1 Score from ~0.39 to ~0.5656).
5. **Prediction & Formatting**: Apply the optimized threshold to the ensemble probabilities on `private_test.csv` and write the outputs in the required submission format.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

### Step 2: Load Data

We read `train.csv`, `public_test.csv`, and `private_test.csv`. Since `public_test.csv` contains the target label `Converted`, we combine it with the training set to maximize the training data volume.

In [ ]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

# Combine train + public test for training
df_all = pd.concat([train, public_test], axis=0, ignore_index=True)
print("Combined Training Dataset Shape:", df_all.shape)

### Step 3: Feature Engineering

We engineer a set of behavioral, interaction, and frequency features designed to capture session intensity, engagement level, purchasing history, and responses to promotions.

In [ ]:
def preprocess_and_engineer(df):
    df = df.copy()
    
    # Missing value indicators
    df['Age_is_missing'] = df['Age'].isna().astype(int)
    df['Income_is_missing'] = df['Income'].isna().astype(int)
    df['Time_On_Site_is_missing'] = df['Time_On_Site'].isna().astype(int)
    
    # Impute missing values with median
    df['Age_imp'] = df['Age'].fillna(df['Age'].median())
    df['Income_imp'] = df['Income'].fillna(df['Income'].median())
    df['Time_On_Site_imp'] = df['Time_On_Site'].fillna(df['Time_On_Site'].median())
    
    # 1. Ratio features
    df['Pages_Per_Product'] = df['Pages_Viewed'] / (df['Products_Viewed'] + 1)
    df['Time_Per_Page'] = df['Time_On_Site_imp'] / (df['Pages_Viewed'] + 1)
    df['Time_Per_Product'] = df['Time_On_Site_imp'] / (df['Products_Viewed'] + 1)
    
    df['Products_Per_Time'] = df['Products_Viewed'] / (df['Time_On_Site_imp'] + 1)
    df['Pages_Per_Time'] = df['Pages_Viewed'] / (df['Time_On_Site_imp'] + 1)
    
    # 2. Income-related
    df['Income_Age_Prod'] = df['Income_imp'] * df['Age_imp']
    df['Income_Age_Ratio'] = df['Income_imp'] / (df['Age_imp'] + 1)
    
    # 3. Shopping engagement score
    df['Engagement_Score'] = df['Pages_Viewed'] * df['Products_Viewed'] * (df['Time_On_Site_imp'] + 1)
    
    # 4. Previous purchases relative to age
    df['Purchases_Per_Age'] = df['Previous_Purchases'] / (df['Age_imp'] + 1)
    
    # 5. Discount interaction features
    df['Discount_Purchases'] = df['Discount_Seen'] * df['Previous_Purchases']
    df['Discount_Income'] = df['Discount_Seen'] * df['Income_imp']
    df['Discount_Time'] = df['Discount_Seen'] * df['Time_On_Site_imp']
    df['Discount_Pages'] = df['Discount_Seen'] * df['Pages_Viewed']
    
    # 6. Frequency encoding of Campaign_Code
    cc_counts = df['Campaign_Code'].value_counts()
    df['Campaign_Code_Freq'] = df['Campaign_Code'].map(cc_counts)
    
    # 7. Convert integer identifiers to string categorical representation
    df['Browser_Version_Cat'] = df['Browser_Version'].astype(str)
    df['City_Tier_Cat'] = df['City_Tier'].astype(str)
    
    return df

# Combine train + test temporarily for consistent encoding
n_train = len(df_all)
df_full = pd.concat([df_all, private_test], axis=0, ignore_index=True)
df_full_engineered = preprocess_and_engineer(df_full)

# Define features to use
features = [
    'Age_imp', 'Income_imp', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site_imp',
    'Previous_Purchases', 'Discount_Seen', 'Device_Type', 'Traffic_Source',
    'Age_is_missing', 'Income_is_missing', 'Time_On_Site_is_missing',
    'Pages_Per_Product', 'Time_Per_Page', 'Time_Per_Product',
    'Products_Per_Time', 'Pages_Per_Time',
    'Income_Age_Prod', 'Income_Age_Ratio', 'Engagement_Score',
    'Purchases_Per_Age', 'Discount_Purchases', 'Discount_Income',
    'Discount_Time', 'Discount_Pages',
    'Browser_Version_Cat', 'City_Tier_Cat', 'Campaign_Code_Freq'
]

# Split back
X_train_full = df_full_engineered.iloc[:n_train][features].copy()
y_train_full = df_full_engineered.iloc[:n_train]['Converted'].copy()
X_private = df_full_engineered.iloc[n_train:][features].copy()

print("Training Features Shape:", X_train_full.shape)
print("Private Test Features Shape:", X_private.shape)

### Step 4: Model Cross-Validation and Training

We use Stratified 5-Fold Cross Validation and train a CatBoost Classifier in each fold. Categorical variables are handled natively by CatBoost.

In [ ]:
# Set up CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Arrays for predictions
oof_preds = np.zeros(len(X_train_full))
private_preds = np.zeros(len(X_private))

# Categorical variables
cat_cols = ['Device_Type', 'Traffic_Source', 'Browser_Version_Cat', 'City_Tier_Cat']

# Prepare data format for CatBoost
X_train_cat = X_train_full.copy()
X_private_cat = X_private.copy()
for col in cat_cols:
    X_train_cat[col] = X_train_cat[col].fillna('missing').astype(str)
    X_private_cat[col] = X_private_cat[col].fillna('missing').astype(str)

print("Training 5-Fold CatBoost Ensemble...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_cat, y_train_full)):
    X_tr, y_tr = X_train_cat.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_cat.iloc[val_idx], y_train_full.iloc[val_idx]
    
    model = CatBoostClassifier(
        random_state=42 + fold,
        verbose=0,
        cat_features=cat_cols,
        iterations=500,
        learning_rate=0.03,
        thread_count=1
    )
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    private_preds += model.predict_proba(X_private_cat)[:, 1] / 5.0
    
    print(f"Fold {fold+1} complete.")

### Step 5: Decision Threshold Optimization

We optimize the decision threshold on the out-of-fold probability pool to maximize the F1 Score.

In [ ]:
# Search for optimal threshold
best_thresh = 0.5
best_f1 = 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y_train_full, (oof_preds > thresh).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"Optimal Decision Threshold: {best_thresh:.2f}")
print(f"Maximum Out-of-Fold F1 Score: {best_f1:.4f}")

# Compute detailed metrics at best threshold
y_pred_best = (oof_preds > best_thresh).astype(int)
print("Precision:", precision_score(y_train_full, y_pred_best))
print("Recall:", recall_score(y_train_full, y_pred_best))
print("Accuracy:", accuracy_score(y_train_full, y_pred_best))
print("F1 Score at Default 0.5 Threshold:", f1_score(y_train_full, (oof_preds > 0.5).astype(int)))

### Step 6: Generate Final Predictions

We average predictions from our 5 fold models, apply our tuned decision threshold, and format the output into `submission.csv`.

In [ ]:
# Apply tuned decision threshold
binary_predictions = (private_preds > best_thresh).astype(int)

# Construct submission DataFrame
submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": binary_predictions
})

# Verify counts and format
print("Submission shape:", submission.shape)
print("Converted Counts:\n", submission["Converted"].value_counts())

# Save submission
submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully!")
submission.head()